# z626 - Regresion Lineal, TODOS los productos y TODOS los meses

## Que cambia respecto a z625 (el original)
`z625` entrenaba con UNA sola foto (periodo 201812) y una lista curada de ~180 "productos magicos". Esta version entrena con TODAS las filas disponibles: cada combinacion (producto, periodo) que tenga los 12 lags completos se convierte en una fila de entrenamiento, para TODOS los productos (no solo la lista curada) y TODOS los periodos historicos (no solo diciembre 2018).

Mismo modelo (OLS, mismos 12 lags como features), pero con muchas mas filas para entrenar.

In [1]:
!pip install -q polars statsmodels

In [2]:
import os
import numpy as np
import polars as pl
import polars.selectors as cs
import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore")

In [3]:
PARAM = {
    'experimento': 'LR02_TODO',
    'kaggle_competition': 'labo-iii-2026-ba',
    'datasets_path': '/home/ds/datasets/',
    'exp_path': '/home/ds/exp/'
}

ruta = os.path.join(PARAM['exp_path'], PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/LR02_TODO


## 1. Cargar TODOS los productos (sin filtrar a apredecir todavia -- mas data para entrenar)

In [4]:
dataset = pl.read_csv(os.path.join(PARAM['datasets_path'], 'sell-in.txt.gz'), separator="\t")

tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])
print(tb_ventas.shape)

(31243, 3)


## 2. Lags y clase (mismo criterio que z625: 12 lags + target a 2 periodos)

In [5]:
lags = [-2, *range(0, 12)]

tb_lags = (
    tb_ventas.sort(["product_id", "periodo"])
    .with_columns(
        [
            pl.col("tn").shift(lag).over("product_id").alias(f"tn_{lag}")
            for lag in lags
        ]
    )
)
tb_lags = tb_lags.rename({"tn_-2": "clase"})
print(tb_lags.shape)

(31243, 16)


## 3. Filtrar a filas con los 12 lags completos y clase no nula (todos los productos, todos los periodos)

In [6]:
campos_lag = [f"tn_{l}" for l in range(0, 12)]

tb_completo = tb_lags.filter(
    pl.all_horizontal([pl.col(c).is_not_null() for c in campos_lag]) & pl.col("clase").is_not_null()
)
print("filas utilizables para entrenar (todos los productos, todos los periodos):", tb_completo.shape)

filas utilizables para entrenar (todos los productos, todos los periodos): (17072, 16)


## 4. Split train / valid (mismo criterio que el resto del proyecto, para comparabilidad)

In [7]:
def periodo_a_meses(periodo: int) -> int:
    return (periodo // 100) * 12 + (periodo % 100)

tb_completo = tb_completo.with_columns(
    (pl.col("periodo").map_elements(periodo_a_meses, return_dtype=pl.Int64) + 2).alias("periodo_target_m")
)

m_201910 = periodo_a_meses(201910)
m_201911 = periodo_a_meses(201911)
m_201912 = periodo_a_meses(201912)

train = tb_completo.filter(pl.col("periodo_target_m") <= m_201910)
valid = tb_completo.filter(
    (pl.col("periodo_target_m") >= m_201911) & (pl.col("periodo_target_m") <= m_201912)
)
print("train:", train.height, " valid:", valid.height)

train: 15575  valid: 1497


## 5. Entrenar OLS con TODOS los productos y periodos disponibles

In [8]:
campos_buenos = train.select(cs.starts_with("tn_"))

X_train = train.select(campos_buenos).to_pandas()
X_train = sm.add_constant(X_train)
y_train = train['clase'].to_pandas()

modelo = sm.OLS(y_train, X_train).fit()
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                  clase   R-squared:                       0.915
Model:                            OLS   Adj. R-squared:                  0.915
Method:                 Least Squares   F-statistic:                 1.404e+04
Date:                Sat, 15 Aug 2026   Prob (F-statistic):               0.00
Time:                        14:45:18   Log-Likelihood:                -76416.
No. Observations:               15575   AIC:                         1.529e+05
Df Residuals:                   15562   BIC:                         1.530e+05
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -1.3394      0.285     -4.695      0.0

## 6. Total Error Rate sobre validacion (antes de predecir el futuro)

In [9]:
X_valid = valid.select(campos_buenos).to_pandas()
X_valid = sm.add_constant(X_valid, has_constant='add')
y_valid = valid['clase'].to_pandas()

pred_valid = modelo.predict(X_valid)
pred_valid = np.clip(pred_valid, 0, None)

total_error_rate = np.abs(pred_valid - y_valid).sum() / y_valid.sum()
print("Total Error Rate (validacion 201911-201912):", total_error_rate)

Total Error Rate (validacion 201911-201912): 6.7612896832785605


## 7. Prediccion para 202002
Solo productos de `apredecir` con los 12 lags completos en 201912; el resto se completa con el promedio del 2019 (mismo fallback que el original).

In [10]:
tb_apredecir = pl.read_csv(os.path.join(PARAM['datasets_path'], 'product_id_apredecir201912.txt'), separator="\t")

dfuture = tb_lags.filter(
    (pl.col("periodo") == 201912) &
    pl.col("product_id").is_in(tb_apredecir["product_id"]) &
    pl.all_horizontal([pl.col(c).is_not_null() for c in campos_lag])
)
print(dfuture.shape)

(656, 16)


In [11]:
X_future = dfuture.select(cs.starts_with("tn_")).to_pandas()
X_future = sm.add_constant(X_future, has_constant='add')

prediccion = modelo.predict(X_future)
prediccion = np.clip(prediccion, 0, None)

tb_regresion = dfuture.select(['product_id']).with_columns(
    pl.Series("tn_pred", prediccion)
)
print(tb_regresion.shape)

(656, 2)


## 8. Fallback (promedio 2019) + submit

In [12]:
tb_meses12 = tb_ventas.filter(pl.col("periodo").is_between(201901, 201912)).group_by("product_id").agg(
    pl.col("tn").mean().alias("tn")
)
tb_meses12 = tb_apredecir.join(tb_meses12, on="product_id", how="left").select(["product_id", "tn"])

tb_final = (
    tb_meses12
    .join(tb_regresion, on="product_id", how="left")
    .with_columns(
        pl.coalesce([pl.col("tn_pred"), pl.col("tn")]).alias("tn")
    )
    .drop("tn_pred")
)
print("nulos en tn (revisar):", tb_final["tn"].null_count())
tb_final = tb_final.with_columns(pl.col("tn").fill_null(0.0))
print(tb_final.shape)
tb_final.head()

nulos en tn (revisar): 0
(780, 2)


product_id,tn
i64,f64
20001,1385.150935
20002,1043.017687
20003,773.857061
20004,580.407273
20005,546.179919


In [13]:
def kaggle_submit(competencia, archivo, mensaje):
    comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
    os.system(comando)

archivo = os.path.join(ruta, "linreg_todo.csv")
tb_final.write_csv(archivo)
print(archivo)

kaggle_submit(PARAM['kaggle_competition'], archivo, "Regresion Lineal - todos los productos y periodos")

/home/ds/exp/LR02_TODO/linreg_todo.csv


100%|██████████| 17.0k/17.0k [00:00<00:00, 52.7kB/s]


96 submissions remaining today.
Successfully submitted to Labo III, 2026 BA